<a href="https://colab.research.google.com/github/SakshamPainuly/underwater-creature-tracking/blob/p4%2Fcolab-baseline/notebooks/colab_train_tritonia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UWCT — Colab / Kaggle YOLO26s training on Tritonia sea slugs (E006, dataset v3)

This notebook only calls repository scripts; every check lives in the code (D-029, D-030, D-033).

- Runtime: **GPU** (Runtime → Change runtime type). Never train on the local PC.
- Dataset **v3**: the 150-frame batch 1, class `1: sea_slug`, split by recording day (10 train /
  3 val / 3 test days, D-030). Its boxes were made by code next to each MTrackJ cross and not
  reviewed (D-033). The training data file has **no test split**.
- **Accuracy is measured against the crosses** (cell 17: point precision, recall, F1, point AP on
  val; graphs and annotated frames in cells 17b and 18b). The P/R/mAP that Ultralytics prints
  during training are against the automatic boxes:
  they are not accuracy and are never reported as such (D-033); cell 17c shows them, labelled so.
- Runs and checkpoints go to `MyDrive/uwct/runs/` on Colab, `/kaggle/working/runs/` on Kaggle
  (outside Git). Every cell raises on failure.
- Frames are 15 s apart: results are sparse-frame/timelapse analysis, never "video tracking".

**Tritonia data on the local PC (D-034):** everything is in one folder, `D:\uwct_tritonia\`:
- `raw\tritonia\`: TIFF stacks + MTrackJ `.mdf` (`UWCT_TRITONIA_ROOT`, read-only)
- `raw\tritonia_boxes_batch1\`: label source of v3 (`UWCT_TRITONIA_BOXES_ROOT`, read-only)
- `work\`: labelling batch, automatic labels, unused hand labels, old label ZIPs

The Drive paths below did not change.

**Kaggle or Colab:** cell 1 detects the platform (`KAGGLE_KERNEL_RUN_TYPE`); nothing else to change.

**Kaggle, first (once):**
1. Private Kaggle dataset `anusidhu7/uwct-tritonia-v3-inputs`, uploaded from the staging copy
   `D:\uwct_tritonia\kaggle_upload\` (copies of the two folders below, hash-checked 2026-11-09):
   `kaggle datasets create -p D:\uwct_tritonia\kaggle_upload --dir-mode zip` (no `--public`).
2. In the notebook: **Add Input** → that dataset (attached read-only under `/kaggle/input`);
   Settings → Accelerator **GPU**, **Internet on**; Add-ons → Secrets → `UWCT_GITHUB_TOKEN`
   (same token as the Colab secret, docs/COLAB.md).
3. `/kaggle/working` is kept only when the notebook version is saved: **Save Version** before the
   session limit. To resume, attach that saved version's output as an input and set
   `KAGGLE_PREVIOUS_RUNS` in cell 1 to its `runs` folder.

**Colab: upload to Google Drive first (once):**
1. `D:\uwct_tritonia\raw\tritonia_boxes_batch1\` (the whole folder, made by `scripts/ingest_tritonia_boxes.py`)
   → `MyDrive/uwct/raw/tritonia_boxes_batch1/`
2. `D:\uwct_tritonia\work\tritonia_label_batch1\` (the batch: `images/`, `points/`, `manifest.csv`,
   `batch.json`; already uploaded 2026-11-03) → `MyDrive/uwct/work/tritonia_label_batch1/`

**First session (no E006 config yet):** run cells 1–10. Cell 9 prints the dataset fingerprint;
put it into `configs/experiments/E006_yolo_seaslug.yaml` on the local PC (D-030 has the text),
commit, push, set `COMMIT` to that commit and run again from cell 1.

In [ ]:
# 1. Settings -- set COMMIT (full 40-character sha of the commit to run); keep everything else.
import os, re, subprocess, sys

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "p4/colab-baseline"  # before PR #16 is merged: "anu/sea-slug-detection"
COMMIT = "0000000000000000000000000000000000000000"  # REQUIRED: `git rev-parse HEAD` on the local PC, after that commit is pushed
DATASET_VERSION = "v3"                                         # frozen Tritonia version: v3 (D-030) or v5 (D-035)
# E009 (dataset v5 = batch 1 + complete frames of batch 2, D-035): DATASET_VERSION = "v5" and
# EXPERIMENT_CONFIG = "configs/experiments/E009_yolo_seaslug_v5.yaml", EXPERIMENT_ID = "E009".
TRITONIA_INPUTS = {  # dataset version -> (Kaggle dataset, label source folder, frame batch folder)
    "v3": ("uwct-tritonia-v3-inputs", "tritonia_boxes_batch1", "tritonia_label_batch1"),
    "v5": ("uwct-tritonia-v5-inputs", "tritonia_boxes_batch2", "tritonia_label_batch2"),
}
KAGGLE_DATASET, BOXES_NAME, BATCH_NAME = TRITONIA_INPUTS[DATASET_VERSION]
EXPERIMENT_CONFIG = "configs/experiments/E006_yolo_seaslug.yaml"
EXPERIMENT_ID = "E006"                                         # must match the config
# E008 (E006 at imgsz 1280, declared in docs/EXPERIMENTS.md): use these two lines instead, then run
# cells 1-13 (smoke), 14 (full), 16, 17, 17a-17c. Cell 18 stays E006 only.
# EXPERIMENT_CONFIG = "configs/experiments/E008_yolo_seaslug_imgsz1280.yaml"
# EXPERIMENT_ID = "E008"
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"  # Colab Secrets / Kaggle Add-ons -> Secrets: see docs/COLAB.md (D-021)

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # detected: Kaggle notebook, otherwise Colab
KAGGLE_PREVIOUS_RUNS = ""  # Kaggle resume only: the runs/ folder of the saved earlier version, under /kaggle/input

WORK = "/tmp" if ON_KAGGLE else "/content"  # scratch space (not saved on Kaggle, keeps the output small)
REPO_DIR = f"{WORK}/uwct"                    # fixed: resumed runs expect the same dataset path
DRIVE_BOXES = f"/content/drive/MyDrive/uwct/raw/{BOXES_NAME}"        # Colab: label source (read-only)
DRIVE_BATCH = f"/content/drive/MyDrive/uwct/work/{BATCH_NAME}"       # Colab: the batch (images/ = 150 clean frames)
RUNS_ROOT = "/kaggle/working/runs" if ON_KAGGLE else "/content/drive/MyDrive/uwct/runs"  # persistent run folders
LOCAL_BOXES = f"{WORK}/raw/{BOXES_NAME}"  # fast local copies
LOCAL_FRAMES = f"{WORK}/tritonia_frames"
LOCAL_BATCH = f"{WORK}/tritonia_batch"
WEIGHTS_DIR = f"{WORK}/weights"                    # pretrained checkpoints

assert re.fullmatch(r"[0-9a-f]{40}", COMMIT) and set(COMMIT) != {"0"}, "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")


def get_secret(name):
    """Read a notebook secret at call time (Kaggle Secrets or Colab Secrets)."""
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    from google.colab import userdata
    return userdata.get(name)


print("platform:", "Kaggle" if ON_KAGGLE else "Colab", "| runs:", RUNS_ROOT)

In [ ]:
# 2. GPU present?
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU -- switch the runtime to GPU")

In [ ]:
# 3. Data and run storage.
#    Colab: mount Google Drive (Tritonia frames + labels, persistent runs folder).
#    Kaggle: find the two folders of the attached dataset under /kaggle/input (read-only; the mount
#    path differs between Kaggle layouts, so the folders are searched, and exactly one must match).
import shutil
if ON_KAGGLE:
    def find_input_dir(name, must_contain):
        hits = sorted(d for d, subdirs, _ in os.walk("/kaggle/input")
                      if os.path.basename(d) == name and must_contain in subdirs and KAGGLE_DATASET in d)
        assert len(hits) == 1, (f"expected one {name}/ folder (with {must_contain}/) of dataset {KAGGLE_DATASET} "
                                f"under /kaggle/input, found {hits} -- attach the dataset (Add Input)")
        return hits[0]
    SRC_BOXES = find_input_dir(BOXES_NAME, "labels")
    SRC_BATCH = find_input_dir(BATCH_NAME, "images")
    if KAGGLE_PREVIOUS_RUNS:  # resume: bring the saved run folders back to the same RUNS_ROOT path
        assert os.path.isdir(KAGGLE_PREVIOUS_RUNS), f"{KAGGLE_PREVIOUS_RUNS} not found -- attach the saved version's output"
        shutil.copytree(KAGGLE_PREVIOUS_RUNS, RUNS_ROOT, dirs_exist_ok=True)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    SRC_BOXES, SRC_BATCH = DRIVE_BOXES, DRIVE_BATCH
SRC_FRAMES = os.path.join(SRC_BATCH, "images")
os.makedirs(RUNS_ROOT, exist_ok=True)
print("label source:", SRC_BOXES, "| frames:", SRC_FRAMES)

In [ ]:
# 4. Check out the exact commit (read-only token from Colab / Kaggle Secrets; see docs/COLAB.md).
#    Git asks the GIT_ASKPASS helper for credentials; the helper answers from the environment of
#    that one git process. The token is read from the notebook secrets inside git_remote() at call time
#    and is never in a command-line argument, URL, file, .git/config or output.
import stat

REPO_URL = f"https://github.com/{REPO}.git"  # no credentials in the URL
ASKPASS = f"{WORK}/uwct_git_askpass.sh"       # holds no secret; printf/echo are shell builtins
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    """Local git commands: no credentials involved."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    """clone/fetch: the token exists only in this git child's environment, for this call only."""
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0",
           "UWCT_GIT_TOKEN": get_secret(TOKEN_SECRET_NAME)}
    try:
        # credential.helper= (empty) disables every credential helper, so nothing is stored
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env,
                           capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        if "403" in err or "not granted" in err:  # token is valid but cannot read this repo
            err += (f"\nThe token in the notebook secret {TOKEN_SECRET_NAME} has no read access to {REPO}."
                    " A fine-grained token only reaches repos owned by its resource owner, so a"
                    " collaborator's token cannot read this personal repo: the repo owner must create"
                    " it (resource owner = repo owner, only this repo, Contents: Read-only)."
                    " See docs/COLAB.md, 'One-time setup'.")
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd=WORK)
else:  # reconnect / new commit: fetch the branch again
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT
assert git("remote", "get-url", "origin") == REPO_URL
print("commit:", COMMIT, "| working tree:", git("status", "--porcelain") or "clean")

In [ ]:
# 5. Install the pinned GPU requirements (ultralytics==8.4.155; torch/CUDA come from Colab)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")
sh(sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, "
   "'| torch', torch.__version__, '| cuda', torch.version.cuda)")

In [ ]:
# 6. Environment report for this session (each run record stores its own copy too)
sh(sys.executable, "scripts/env_report.py", "--out", f"{WORK}/env_report_session.json")

In [ ]:
# 7. Copy the Tritonia frames and label source from Drive / Kaggle input to local disk. Nothing is
#    written to the source.
#    The label source is registered as a read-only raw root (UWCT_TRITONIA_BOXES_ROOT): scripts refuse
#    to write inside it.
for src, dst in ((SRC_BOXES, LOCAL_BOXES), (SRC_FRAMES, LOCAL_FRAMES)):
    assert os.path.isdir(src), f"{src} not found -- upload / attach it first (see the top of this notebook)"
    if os.path.isdir(dst):
        shutil.rmtree(dst)  # start clean
    shutil.copytree(src, dst)
os.environ["UWCT_TRITONIA_BOXES_ROOT"] = LOCAL_BOXES  # read by every later script
print(len(os.listdir(os.path.join(LOCAL_BOXES, "labels"))), "label files |",
      len([f for f in os.listdir(LOCAL_FRAMES) if f.endswith(".png")]), "frames")

In [ ]:
# 8. Raw-data check: the frozen v3 manifests record the sha256 of every frame and label file, and the
#    label source's ingest_report.json. Cell 9 refuses to build if any of them differs, so the build is
#    the content check of this local copy (CLAUDE.md section 6).
import json
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from dataset_audit.versioning import read_version_meta, verify_frozen
verify_frozen(DATASET_VERSION)
meta = read_version_meta(DATASET_VERSION)
assert meta["dataset_kind"] == "tritonia", meta
print(json.dumps({k: meta[k] for k in ("name", "decision", "status", "class_names", "split_rule")}, indent=2))

In [ ]:
# 9. Build the YOLO view of v3 (copies, every file hash-checked) and print its dataset fingerprint.
sh(sys.executable, "scripts/prepare_tritonia_yolo.py", "--version", DATASET_VERSION,
   "--frames", LOCAL_FRAMES, "--boxes", LOCAL_BOXES)
DATASET_DIR = f"data/yolo/tritonia_{DATASET_VERSION}"
BUILT_FINGERPRINT = json.load(open(os.path.join(REPO_DIR, DATASET_DIR, "build_summary.json")))["dataset_fingerprint"]
print("dataset_fingerprint for E006:", BUILT_FINGERPRINT)

In [ ]:
# 10. Ultralytics reads the view exactly as built (names {0: lobster, 1: sea_slug}). Must print VERIFIED.
sh(sys.executable, "scripts/verify_yolo_dataset.py", "--dataset", DATASET_DIR)

In [ ]:
# 11. The experiment config exists at COMMIT and names this dataset and fingerprint.
from training.experiment import load_config
config_path = os.path.join(REPO_DIR, EXPERIMENT_CONFIG)
assert os.path.isfile(config_path), (f"{EXPERIMENT_CONFIG} is not in this commit yet. Create it on the local PC "
                                     f"with dataset_fingerprint: \"{BUILT_FINGERPRINT}\" (template in D-030), "
                                     "commit, push, set COMMIT and start again from cell 1.")
cfg = load_config(config_path)
assert cfg["id"] == EXPERIMENT_ID and cfg["dataset_version"] == DATASET_VERSION, cfg
assert cfg["dataset_fingerprint"] == BUILT_FINGERPRINT, (cfg["dataset_fingerprint"], BUILT_FINGERPRINT)
print("config OK:", cfg["id"], cfg["model"], cfg["train_args"])

In [ ]:
# 12. Preflight: every training check + load (download) the exact pretrained checkpoint.
TRAIN = [sys.executable, "scripts/train_yolo.py", "--config", EXPERIMENT_CONFIG,
         "--runs-root", RUNS_ROOT, "--weights-dir", WEIGHTS_DIR]
sh(*TRAIN, "--preflight-only")

In [ ]:
# 13. Smoke test: 1 epoch on 10% of train, validated on val -> RUNS_ROOT/E006_smoke_<utc>/
RUN_SMOKE = False  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled"
sh(*TRAIN, "--smoke")

In [ ]:
# 14. Full training run -> RUNS_ROOT/E006/. Only after the smoke test passed and the run is approved.
RUN_FULL = False  # set to True deliberately
assert RUN_FULL, "Full training not enabled"
sh(*TRAIN)

In [ ]:
# 15. Resume after a Colab disconnect: re-run cells 1-11 (same COMMIT), then this cell.
#     Kaggle: new session with the saved version's output attached and KAGGLE_PREVIOUS_RUNS set (cell 1).
RESUME = False  # set to True deliberately
assert RESUME, "Resume not enabled"
sh(*TRAIN, "--resume")

In [ ]:
# 16. Summary of the run record. Then download run_record.json and, on the local PC:
#     python scripts/register_run.py <path>/run_record.json
rec = json.load(open(os.path.join(RUNS_ROOT, EXPERIMENT_ID, "run_record.json"), encoding="utf-8"))
print(json.dumps({k: rec[k] for k in ("run_name", "git", "seed", "ultralytics_version", "val_metrics")}, indent=2))
print("dataset:", rec["dataset"]["version"], rec["dataset"]["fingerprint"])
print("best:", rec["outputs"]["best"])

In [ ]:
# 17. Val point scores against the MTrackJ crosses (D-033): the accuracy of E006. Val only.
#     Runs best.pt on every frame of the 3 val days (frames the automatic labels excluded too),
#     matches boxes to crosses (10 px, one to one) and chooses the threshold with the best val F1.
#     Writes new files in the run folder; refuses to overwrite an earlier result.
if os.path.isdir(LOCAL_BATCH):
    shutil.rmtree(LOCAL_BATCH)
shutil.copytree(SRC_BATCH, LOCAL_BATCH, ignore=shutil.ignore_patterns("hints"))
best = rec["outputs"]["best"]["path"]
run_dir = os.path.join(RUNS_ROOT, EXPERIMENT_ID)
sh(sys.executable, "scripts/tritonia_point_eval.py", "--batch", LOCAL_BATCH, "--split", "val",
   "--weights", best, "--imgsz", str(cfg["train_args"]["imgsz"]),
   "--pred-dir", os.path.join(run_dir, "val_point_predictions"),
   "--out", os.path.join(run_dir, "val_point_scores.json"))

In [ ]:
# 17a. Report helpers -- run once per session before 17b, 17c or 18b (after cells 1-11; no training).
#      Finds E006's files in RUNS_ROOT, or on Kaggle in an attached saved version under /kaggle/input.
import glob
from IPython.display import Image, Markdown, display


def find_run_file(rel):
    """RUNS_ROOT/<rel> if it exists, else exactly one match under /kaggle/input (Kaggle only)."""
    p = os.path.join(RUNS_ROOT, rel)
    if os.path.exists(p):
        return p
    hits = sorted(glob.glob(f"/kaggle/input/**/{rel}", recursive=True)) if ON_KAGGLE else []
    assert len(hits) == 1, (f"{rel}: not in {RUNS_ROOT}" + (f" and {len(hits)} matches under /kaggle/input {hits}"
                            if ON_KAGGLE else "") + " -- run the cell that makes it, or attach the saved version (Add Input)")
    return hits[0]


def ensure_batch():
    """Local copy of the labelling batch (frames + crosses), as in cells 17 and 18."""
    if not os.path.isdir(LOCAL_BATCH):
        shutil.copytree(SRC_BATCH, LOCAL_BATCH, ignore=shutil.ignore_patterns("hints"))


def show_images(paths, width=900):
    for p in paths:
        display(Markdown(f"`{os.path.basename(p)}`"))
        display(Image(filename=p, width=width))


def show_point_report(report_dir, frames_to_show=None):
    """Tables, graphs and annotated frames written by scripts/tritonia_point_report.py."""
    rep = json.load(open(os.path.join(report_dir, "report.json"), encoding="utf-8"))
    a = rep["at_conf"]
    rows = ["| | predictions | crosses | matched | point precision | point recall | point F1 |",
            "|---|---:|---:|---:|---:|---:|---:|",
            f"| **all {rep['frames']} frames** | {a['predictions']} | {a['crosses']} | {a['matched']} | "
            f"{a['precision']:.3f} | {a['recall']:.3f} | {a['f1']:.3f} |"]
    rows += [f"| {day} | {s['predictions']} | {s['crosses']} | {s['matched']} | {s['precision']:.3f} | "
             f"{s['recall']:.3f} | {s['f1']:.3f} |" for day, s in rep["per_day"].items()]
    display(Markdown(f"## E006 point scores vs the MTrackJ crosses -- `{rep['split']}` (D-033)\n"
                     f"Threshold conf **{rep['conf']:.2f}** ({rep['conf_source']}); match within "
                     f"{rep['match_px']:g} px, one to one. **Point AP {rep['point_ap']:.3f}** (not mAP).\n\n"
                     + "\n".join(rows)))
    for name in ("point_confusion_matrix.png", "point_pr_curve.png", "point_conf_curves.png", "point_per_day.png"):
        display(Image(filename=os.path.join(report_dir, name), width=650))
    frames = sorted(glob.glob(os.path.join(report_dir, "frames", "*.jpg")))
    shown = frames if frames_to_show is None else frames[:frames_to_show]
    display(Markdown(f"### Frames with predictions ({len(shown)} of {len(frames)}; all are in `{report_dir}/frames`)\n"
                     "green = box matched to a cross, red = false box (no cross within 10 px), "
                     "cyan circle = found cross, yellow X = missed cross"))
    show_images(shown, width=1000)
    display(Markdown(f"**LIMITATION:** {rep['limitation']} 27 frames per split: per-day numbers are noisy."))


print("helpers ready | runs:", RUNS_ROOT)

In [ ]:
# 17b. VAL report (D-033): graphs + every val frame with boxes and crosses, from cell 17's files.
#      Scores nothing new: the threshold comes from val_point_scores.json, and the script refuses if the
#      predictions do not reproduce those scores. Writes RUNS_ROOT/E006_val_point_report/ (replaced on re-run).
ensure_batch()
val_scores_path = find_run_file(f"{EXPERIMENT_ID}/val_point_scores.json")
VAL_REPORT_DIR = os.path.join(RUNS_ROOT, f"{EXPERIMENT_ID}_val_point_report")
sh(sys.executable, "scripts/tritonia_point_report.py", "--batch", LOCAL_BATCH, "--scores", val_scores_path,
   "--pred-dir", find_run_file(f"{EXPERIMENT_ID}/val_point_predictions"), "--out", VAL_REPORT_DIR, "--replace")
show_point_report(VAL_REPORT_DIR, frames_to_show=8)

In [ ]:
# 17c. Ultralytics box metrics on VAL: mAP50, mAP50-95, confusion matrix, curves, training graphs.
#      NOT ACCURACY (D-033): Ultralytics scores against the AUTOMATIC boxes, which exist for only about half
#      of the crosses (val: 73 boxes for 153 crosses) and sometimes sit on texture. A correct detection of a
#      slug that got no automatic box counts as a false positive here. Accuracy = cells 17 / 17b.
#      Reads the files training wrote (val_best/ = best.pt validated after training); runs nothing.
rec_box = json.load(open(find_run_file(f"{EXPERIMENT_ID}/run_record.json"), encoding="utf-8"))
vm = rec_box["val_metrics"]
display(Markdown(
    "## E006 box metrics vs automatic boxes (val) -- NOT accuracy (D-033)\n"
    "| metric (Ultralytics, class sea_slug) | value |\n|---|---:|\n"
    f"| precision | {vm['precision']:.3f} |\n| recall | {vm['recall']:.3f} |\n"
    f"| mAP50 | {vm['map50']:.3f} |\n| mAP50-95 | {vm['map50_95']:.3f} |\n\n"
    f"{vm.get('note', '')} Not comparable with the lobster mAP of E001-E005 (other data, automatic boxes)."))
val_best = find_run_file(f"{EXPERIMENT_ID}/val_best")
train_dir = find_run_file(f"{EXPERIMENT_ID}/train")
for title, names in (("Confusion matrix (rows = predicted, cols = automatic boxes)",
                      ["confusion_matrix.png", "confusion_matrix_normalized.png"]),
                     ("Precision-recall curve (mAP50)", ["BoxPR_curve.png"]),
                     ("F1 / precision / recall vs confidence", ["BoxF1_curve.png", "BoxP_curve.png", "BoxR_curve.png"])):
    found = [os.path.join(val_best, n) for n in names if os.path.isfile(os.path.join(val_best, n))]
    display(Markdown(f"### {title}" + ("" if found else " -- not produced by this Ultralytics version")))
    for p in found:
        display(Image(filename=p, width=700))
display(Markdown("### Training graphs (losses and val metrics per epoch, `train/results.png`)"))
show_images([os.path.join(train_dir, "results.png")], width=1000)
display(Markdown("### Val batches: automatic boxes (`_labels`) vs predictions with confidence (`_pred`)"))
show_images(sorted(glob.glob(os.path.join(val_best, "val_batch*_*.jpg"))), width=1000)

In [ ]:
# 18. FINAL TEST of the declared candidate E006 -- run ONCE (rule 7, D-033). New session: Add Input ->
#     the saved notebook version whose output holds runs/E006 (best.pt + val_point_scores.json); run
#     cells 1-11 only (no training), then this cell. The threshold is the val one (cell 17); test never
#     changes it. Writes RUNS_ROOT/E006_final_test/; the script refuses to overwrite an earlier result.
import csv, glob, hashlib
RUN_FINAL_TEST = False  # set to True deliberately, once
assert RUN_FINAL_TEST, "Final test not enabled"
FINAL_ID = "E006"
FINAL_CONF = 0.06  # best val F1 threshold of E006 (val_point_scores.json, docs/EXPERIMENTS.md)
FINAL_BEST_SHA256 = "6f6710d1ed2e10408a01012b43a2a7979f5d739b07a8e03f52ba3fa15a4d6c8c"  # registered best.pt

with open(os.path.join(REPO_DIR, "experiments", "final_candidates.csv"), encoding="utf-8") as f:
    declared = {r["experiment_id"] for r in csv.DictReader(f) if r["decision"] == "candidate"}
assert FINAL_ID in declared, f"{FINAL_ID} is not declared in experiments/final_candidates.csv at this commit"


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


roots = ["/kaggle/input"] if ON_KAGGLE else [RUNS_ROOT]
found = [p for r in roots for p in glob.glob(f"{r}/**/{FINAL_ID}/train/weights/best.pt", recursive=True)]
best_final = [p for p in found if sha256(p) == FINAL_BEST_SHA256]
assert best_final, (f"no best.pt with sha256 {FINAL_BEST_SHA256[:12]}... under {roots} (found {found}) -- "
                    "attach the saved version's output (Add Input)")
best_final = best_final[0]
val_scores = json.load(open(os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(best_final))),
                                         "val_point_scores.json"), encoding="utf-8"))
assert val_scores["split"] == "val" and val_scores["best_f1_on_val"]["conf"] == FINAL_CONF, val_scores["best_f1_on_val"]
print("weights:", best_final, "| sha256 OK | val threshold:", FINAL_CONF)

if os.path.isdir(LOCAL_BATCH):
    shutil.rmtree(LOCAL_BATCH)
shutil.copytree(SRC_BATCH, LOCAL_BATCH, ignore=shutil.ignore_patterns("hints"))
test_dir = os.path.join(RUNS_ROOT, f"{FINAL_ID}_final_test")
os.makedirs(test_dir, exist_ok=False)  # once: an earlier final test stops here
sh(sys.executable, "scripts/tritonia_point_eval.py", "--batch", LOCAL_BATCH, "--split", "test",
   "--final", FINAL_ID, "--conf", str(FINAL_CONF), "--weights", best_final,
   "--imgsz", str(cfg["train_args"]["imgsz"]),
   "--pred-dir", os.path.join(test_dir, "test_point_predictions"),
   "--out", os.path.join(test_dir, "test_point_scores.json"))
print("commit:", COMMIT, "| Save Version (Quick Save, always save output) before closing the session")

In [ ]:
# 18b. TEST report of the final evaluation: graphs + every test frame with boxes and crosses.
#      Run after cell 18 (same session, or a later one with that saved version attached), after 17a.
#      Reads cell 18's files only: no new test evaluation, the threshold stays the val one (0.06).
#      Box mAP on test is not computed (the declared final test is the point eval, D-033).
#      Writes RUNS_ROOT/E006_final_test_report/. Log the test numbers from cell 18 in docs/EXPERIMENTS.md.
ensure_batch()
test_scores_path = find_run_file(f"{EXPERIMENT_ID}_final_test/test_point_scores.json")
_t = json.load(open(test_scores_path, encoding="utf-8"))
assert _t["split"] == "test" and _t["final"] == EXPERIMENT_ID and _t["at_conf"]["conf"] == 0.06, _t["at_conf"]
TEST_REPORT_DIR = os.path.join(RUNS_ROOT, f"{EXPERIMENT_ID}_final_test_report")
sh(sys.executable, "scripts/tritonia_point_report.py", "--batch", LOCAL_BATCH, "--scores", test_scores_path,
   "--pred-dir", find_run_file(f"{EXPERIMENT_ID}_final_test/test_point_predictions"), "--out", TEST_REPORT_DIR,
   "--replace")
show_point_report(TEST_REPORT_DIR, frames_to_show=8)
print("Save Version (Quick Save, always save output) to keep the report")

## Final test (cell 18)

- Test is scored only for a declared final candidate (`experiments/final_candidates.csv`), with
  the threshold chosen on val (cell 17's `best_f1_on_val`), once, and logged in `docs/EXPERIMENTS.md`
  (CLAUDE.md rule 7; the script refuses test without `--final` and `--conf`). E006 was declared on
  2026-11-10 with conf 0.06; cell 18 checks the declaration, the `best.pt` sha256 and the threshold.

- Report E006 with point precision, recall, F1 and point AP on val, overall and per day, never a
  single number. No mAP50/mAP50-95 for Tritonia: that needs hand-drawn boxes (D-033). Point
  precision is a lower bound (it is UNKNOWN whether every slug was clicked). v3 is 150 frames from
  one setup, so val and test measure new days, not new sites.

## Reports (cells 17a-17c, 18b)

- **17a** defines helpers; run it once per session after cells 1-11.
- **17b** (val) and **18b** (test, after cell 18): point precision-recall curve (area = point AP),
  P / R / F1 against the confidence threshold, per-day bars, a point confusion matrix (matched
  boxes, false boxes, missed crosses; there is no true-negative cell in detection) and every frame
  with its boxes and crosses (`scripts/tritonia_point_report.py`). They score nothing new.
- **17c**: Ultralytics' val mAP50, mAP50-95, confusion matrix, curves and training graphs. These
  are against the automatic boxes, not the crosses: **not accuracy** (D-033), and not
  comparable with the lobster mAP of E001-E005.